In [31]:
pip install meshio imageio matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


In [32]:

# Point this at the C++ Chaste output folder
INPUT_DIR = r"C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50"

import os
if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(f"Output directory not found: {INPUT_DIR}")

print(f"Using output folder:\n  {INPUT_DIR}")


Using output folder:
  C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50


In [33]:
import os
import glob
import csv
import re

import numpy as np
import meshio
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ----------------------------------------------------------------------
# 1. PARAMETERS
# ----------------------------------------------------------------------

FRAME_DIR = os.path.join(INPUT_DIR, "jamming_frames")
VEL_FRAME_DIR = os.path.join(INPUT_DIR, "velocity_frames")
os.makedirs(FRAME_DIR, exist_ok=True)
os.makedirs(VEL_FRAME_DIR, exist_ok=True)

GIF_JAMMING = os.path.join(INPUT_DIR, "jamming_animation.gif")
GIF_VELOCITY = os.path.join(INPUT_DIR, "velocity_animation.gif")
CSV_STATS = os.path.join(INPUT_DIR, "jamming_stats.csv")
PNG_HIST_HEATMAP = os.path.join(INPUT_DIR, "shape_index_hist_heatmap.png")
PNG_TRANSITION_MAP = os.path.join(INPUT_DIR, "jamming_transition_map.png")

# Critical shape index for jamming transition (Bi et al. 2015)
SHAPE_INDEX_CRIT = 3.81

# Fixed box size — 15x4 rectangle
X_MIN, X_MAX = 0.0, 15.0
Y_MIN, Y_MAX = 0.0, 4.0

# Colors:
JAMMED_COLOR = "#b5e7a0"   # light green
UNJAMMED_COLOR = "#ffb3b3" # light red
BOUNDARY_COLOR = "#e0e0e0" # light gray (excluded from bulk stats)

# Histogram settings for shape index heatmap
SHAPE_Q_MIN, SHAPE_Q_MAX = 3.0, 5.0
SHAPE_Q_BINS = 40  


# ----------------------------------------------------------------------
# 2. GENERIC HELPERS
# ----------------------------------------------------------------------

def collect_vtu_files(base_dir):
    time_dirs = glob.glob(os.path.join(base_dir, "results_from_time_*"))
    snapshots = []
    for td in time_dirs:
        m = re.search(r"results_from_time_(\d+)", os.path.basename(td))
        if not m: continue
        hour = int(m.group(1))
        vtu = os.path.join(td, "results_2000.vtu")
        if os.path.isfile(vtu):
            snapshots.append((hour, vtu))
    snapshots.sort(key=lambda x: x[0])
    return snapshots

def polygon_area(pts):
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def polygon_perimeter(pts):
    dx = np.diff(np.r_[pts[:, 0], pts[0, 0]])
    dy = np.diff(np.r_[pts[:, 1], pts[0, 1]])
    return np.sum(np.hypot(dx, dy))

def polygon_centroid(pts):
    return pts.mean(axis=0)


# ----------------------------------------------------------------------
# 3. VTU LOADING
# ----------------------------------------------------------------------

def load_cells_from_vtu(vtu_path):
    mesh = meshio.read(vtu_path)
    points = mesh.points[:, :2] 
    cell_polys = []
    for cell_block in mesh.cells:
        if cell_block.type in ("polygon", "vtk_polygon"):
            for poly in cell_block.data:
                cell_polys.append(poly)
    if not cell_polys:
        raise ValueError(f"Could not find polygonal cells in {vtu_path}.")
    return points, cell_polys


# ----------------------------------------------------------------------
# 4. QUANTITIES & TOPOLOGY PER FRAME
# ----------------------------------------------------------------------

def compute_shape_index_for_frame(points, cell_polys):
    shape_index = np.zeros(len(cell_polys), dtype=float)
    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        A = polygon_area(pts)
        P = polygon_perimeter(pts)
        shape_index[i] = P / np.sqrt(A)
    return shape_index

def compute_centroids(points, cell_polys):
    centroids = np.zeros((len(cell_polys), 2), dtype=float)
    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        centroids[i] = polygon_centroid(pts)
    return centroids

def classify_jammed(shape_index):
    return shape_index < SHAPE_INDEX_CRIT

def build_adjacency(cell_polys):
    edge_to_cells = {}
    for cell_idx, poly in enumerate(cell_polys):
        verts = list(poly)
        n = len(verts)
        for k in range(n):
            v1 = verts[k]
            v2 = verts[(k + 1) % n]
            edge = frozenset({v1, v2})
            if edge not in edge_to_cells:
                edge_to_cells[edge] = set()
            edge_to_cells[edge].add(cell_idx)

    neighbors = set()
    for cells in edge_to_cells.values():
        cells = list(cells)
        if len(cells) == 2:
            neighbors.add(frozenset({cells[0], cells[1]}))
    return neighbors

def identify_boundary_cells(cell_polys):
    """
    Identifies cells that have at least one unshared edge (boundary cells).
    Returns a boolean array where True = boundary cell.
    """
    edge_to_cells = {}
    for cell_idx, poly in enumerate(cell_polys):
        verts = list(poly)
        n = len(verts)
        for k in range(n):
            v1 = verts[k]
            v2 = verts[(k + 1) % n]
            edge = frozenset({v1, v2})
            if edge not in edge_to_cells:
                edge_to_cells[edge] = set()
            edge_to_cells[edge].add(cell_idx)

    boundary_cells = set()
    for cells in edge_to_cells.values():
        if len(cells) == 1:
            boundary_cells.add(list(cells)[0])

    is_boundary = np.zeros(len(cell_polys), dtype=bool)
    is_boundary[list(boundary_cells)] = True
    return is_boundary


# ----------------------------------------------------------------------
# 5. PLOTTING FUNCTIONS
# ----------------------------------------------------------------------

def plot_jamming_frame(points, cell_polys, jammed_mask, is_boundary, shape_index, hour, save_path):
    fig, ax = plt.subplots(figsize=(12, 4))

    for i, poly in enumerate(cell_polys):
        pts = points[poly]

        if is_boundary[i]:
            facecolor = BOUNDARY_COLOR
        elif jammed_mask[i]:
            facecolor = JAMMED_COLOR
        else:
            facecolor = UNJAMMED_COLOR

        ax.fill(
            pts[:, 0], pts[:, 1],
            facecolor=facecolor,
            edgecolor="k", linewidth=0.3, alpha=0.9,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    # Only compute statistics for bulk cells
    bulk_q = shape_index[~is_boundary]
    bulk_jammed = jammed_mask[~is_boundary]
    
    mean_q = bulk_q.mean() if len(bulk_q) > 0 else np.nan
    frac_jammed = np.mean(bulk_jammed) if len(bulk_jammed) > 0 else np.nan

    ax.set_title(
        f"t = {hour} h, ⟨q_bulk⟩ = {mean_q:.2f}, "
        f"bulk jammed frac = {frac_jammed:.2f}"
    )

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def plot_velocity_frame(points, cell_polys, centroids, velocities, is_boundary, shape_index, hour, save_path):
    fig, ax = plt.subplots(figsize=(12, 4))

    # Use a continuous colormap for the Shape Index (q)
    # 3.81 is the transition. We map 3.6 (solid core) to 4.1 (highly fluid leading edge)
    vmin, vmax = 3.6, 4.1
    cmap = plt.cm.plasma  

    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        
        if is_boundary[i]:
            facecolor = BOUNDARY_COLOR # Keep boundary cells neutral gray
        else:
            q = shape_index[i]
            # Normalize q to a 0.0 - 1.0 scale for the colormap
            norm_q = np.clip((q - vmin) / (vmax - vmin), 0.0, 1.0)
            facecolor = cmap(norm_q)

        ax.fill(
            pts[:, 0], pts[:, 1],
            facecolor=facecolor, edgecolor="k", linewidth=0.3, alpha=0.9,
        )

    # Overlay Velocity vectors on top
    vx = velocities[:, 0]
    vy = velocities[:, 1]
    
    # White arrows with black outlines stand out against both dark purple and bright yellow
    ax.quiver(
        centroids[:, 0], centroids[:, 1], vx, vy,
        angles="xy", scale_units="xy", scale=1.0, width=0.003, 
        color="white", edgecolor="black", linewidth=0.5
    )

    speeds = np.linalg.norm(velocities, axis=1)
    mean_speed = np.nanmean(speeds)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    
    # Add a dedicated colorbar for the Shape Index
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.04)
    cbar.set_label("Shape Index (q) - Fluidity")
    
    # Mark the exact 3.81 jamming threshold on the colorbar
    threshold_norm = (3.81 - vmin) / (vmax - vmin)
    cbar.ax.axhline(threshold_norm, color='white', linestyle='--', linewidth=1.5)
    cbar.ax.text(1.5, threshold_norm, '3.81 (Jamming)', va='center', ha='left', color='black', fontsize=9)

    ax.set_title(f"Kinematics vs. Fluidity at t = {hour} h, ⟨speed⟩ ≈ {mean_speed:.3f}")

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def plot_shape_index_heatmap(hist_matrix, bin_edges):
    n_frames, n_bins = hist_matrix.shape
    q_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        hist_matrix, aspect="auto", origin="lower",
        extent=[q_centers[0], q_centers[-1], 0, n_frames - 1],
        interpolation="nearest",
    )
    ax.set_xlabel("Bulk Shape index q")
    ax.set_ylabel("Hour")
    ax.set_title("Bulk Shape Index distribution over time")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Bulk Cell count per bin")
    plt.tight_layout()
    fig.savefig(PNG_HIST_HEATMAP, dpi=150)
    plt.close(fig)


def plot_transition_map(points_last, cell_polys_last, jam_history, n_frames, is_boundary_last):
    n_cells_last = len(cell_polys_last)
    first_unjam = np.zeros(n_cells_last, dtype=float)

    for cell_idx in range(n_cells_last):
        history = jam_history.get(cell_idx, [])
        prev = None
        first = None
        for f, val in enumerate(history):
            if val is None:
                continue
            if val is False and (prev is None or prev is True):
                first = f
                break
            prev = val
            
        if first is None:
            first_unjam[cell_idx] = 0.0  
        else:
            first_unjam[cell_idx] = float(first + 1) 

    fig, ax = plt.subplots(figsize=(12, 4))
    vmax = max(1.0, first_unjam.max())

    for i, poly in enumerate(cell_polys_last):
        pts = points_last[poly]
        
        if is_boundary_last[i]:
            # Leave final boundary cells grey
            ax.fill(pts[:, 0], pts[:, 1], facecolor=BOUNDARY_COLOR, edgecolor="k", linewidth=0.3, alpha=0.95)
        else:
            value = first_unjam[i]
            color_val = value / vmax if vmax > 0 else 0.0
            ax.fill(pts[:, 0], pts[:, 1], facecolor=plt.cm.viridis(color_val), edgecolor="k", linewidth=0.3, alpha=0.95)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)

    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=plt.Normalize(vmin=0, vmax=vmax), cmap="viridis"),
        ax=ax,
    )
    cbar.set_label("Hour of first bulk unjamming (0 = never)")

    ax.set_title("Bulk Jamming → Unjamming transition map")
    plt.tight_layout()
    fig.savefig(PNG_TRANSITION_MAP, dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------
# 6. MAIN LOOP OVER TIMESTEPS
# ----------------------------------------------------------------------

def main():
    snapshots = collect_vtu_files(INPUT_DIR)
    if not snapshots: raise FileNotFoundError(f"No VTU snapshots found.")

    print(f"Found {len(snapshots)} hourly VTU snapshots")

    frame_paths_jam = []
    frame_paths_vel = []
    stats_rows = []

    hist_rows = []
    bin_edges = np.linspace(SHAPE_Q_MIN, SHAPE_Q_MAX, SHAPE_Q_BINS + 1)

    prev_centroids = None
    prev_neighbors = None
    jam_history = {}

    last_points = None
    last_cell_polys = None
    last_is_boundary = None

    for frame_idx, (hour, vtu_path) in enumerate(snapshots):
        print(f"Processing hour={hour} ({vtu_path})")

        points, cell_polys = load_cells_from_vtu(vtu_path)
        is_boundary = identify_boundary_cells(cell_polys)
        
        shape_index = compute_shape_index_for_frame(points, cell_polys)
        centroids = compute_centroids(points, cell_polys)
        jammed_mask = classify_jammed(shape_index)

        # Update jamming history (None for boundary cells so they don't trigger transitions)
        for cell_idx in range(len(cell_polys)):
            if cell_idx not in jam_history:
                jam_history[cell_idx] = [None] * frame_idx
            if is_boundary[cell_idx]:
                jam_history[cell_idx].append(None)
            else:
                jam_history[cell_idx].append(bool(jammed_mask[cell_idx]))

        # Histogram row (BULK ONLY)
        bulk_shape_index = shape_index[~is_boundary]
        hist, _ = np.histogram(bulk_shape_index, bins=bin_edges, range=(SHAPE_Q_MIN, SHAPE_Q_MAX))
        hist_rows.append(hist)

        # Velocities
        if prev_centroids is None:
            velocities = np.zeros_like(centroids)
            mean_speed = 0.0
        else:
            n_prev, n_curr = prev_centroids.shape[0], centroids.shape[0]
            n_common = min(n_prev, n_curr)
            velocities = np.zeros_like(centroids)
            velocities[:n_common] = centroids[:n_common] - prev_centroids[:n_common]
            speeds = np.linalg.norm(velocities, axis=1)
            mean_speed = float(np.nanmean(speeds))

        # T1 transitions
        neighbors = build_adjacency(cell_polys)
        if prev_neighbors is None:
            t1_count = 0
        else:
            sym_diff = neighbors ^ prev_neighbors
            t1_count = len(sym_diff) // 2

        # Store stats (BULK ONLY for Q and Jamming)
        mean_q = float(bulk_shape_index.mean()) if len(bulk_shape_index) > 0 else np.nan
        bulk_jammed = jammed_mask[~is_boundary]
        frac_jammed = float(np.mean(bulk_jammed)) if len(bulk_jammed) > 0 else np.nan

        stats_rows.append([
            frame_idx, hour, os.path.basename(vtu_path),
            mean_q, frac_jammed, mean_speed, t1_count,
        ])

        # Plots
        frame_jam_path = os.path.join(FRAME_DIR, f"frame_{frame_idx:04d}.png")
        plot_jamming_frame(points, cell_polys, jammed_mask, is_boundary, shape_index, hour, frame_jam_path)
        frame_paths_jam.append(frame_jam_path)

        frame_vel_path = os.path.join(VEL_FRAME_DIR, f"frame_{frame_idx:04d}.png")
        plot_velocity_frame(points, cell_polys, centroids, velocities, is_boundary, shape_index, hour, frame_vel_path)
        frame_paths_vel.append(frame_vel_path)

        # Update state
        prev_centroids = centroids
        prev_neighbors = neighbors
        last_points = points
        last_cell_polys = cell_polys
        last_is_boundary = is_boundary

    # Save outputs
    with open(CSV_STATS, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame_idx", "hour", "file", "mean_bulk_shape_index", "fraction_bulk_jammed", "mean_speed", "approx_T1_count"])
        writer.writerows(stats_rows)

    imageio.mimsave(GIF_JAMMING, [imageio.imread(p) for p in frame_paths_jam], fps=15)
    imageio.mimsave(GIF_VELOCITY, [imageio.imread(p) for p in frame_paths_vel], fps=15)

    hist_matrix = np.array(hist_rows, dtype=float)
    plot_shape_index_heatmap(hist_matrix, bin_edges)

    if last_points is not None and last_cell_polys is not None:
        plot_transition_map(last_points, last_cell_polys, jam_history, len(snapshots), last_is_boundary)

    print(f"\nSaved stats CSV to       : {CSV_STATS}")
    print(f"Saved jamming GIF to    : {GIF_JAMMING}")
    print(f"Saved velocity GIF to   : {GIF_VELOCITY}")
    print(f"Saved hist heatmap to   : {PNG_HIST_HEATMAP}")
    print(f"Saved transition map to : {PNG_TRANSITION_MAP}")


if __name__ == "__main__":
    main()

Found 153 hourly VTU snapshots
Processing hour=0 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_0\results_2000.vtu)


C:\Users\cmchu\AppData\Local\Temp\ipykernel_19120\1874130817.py:267: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Processing hour=1 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_1\results_2000.vtu)
Processing hour=2 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_2\results_2000.vtu)
Processing hour=3 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_3\results_2000.vtu)
Processing hour=4 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_4\results_2000.vtu)
Processing hour=5 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50\results_from_time_5\results_2000.vtu)
Processing hour=6 (C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale

In [34]:

# ----------------------------------------------------------------------
# SEPARATE BLOCK: GENERIC BEIGE MESH ANIMATION
# Run this in a NEW cell after the main analysis code.
# Uses collect_vtu_files() and load_cells_from_vtu() from Cell 3.
# ----------------------------------------------------------------------

import os
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np

# Output directory & files
GENERIC_FRAME_DIR = os.path.join(INPUT_DIR, "generic_frames")
GENERIC_GIF = os.path.join(INPUT_DIR, "generic_mesh_animation.gif")
os.makedirs(GENERIC_FRAME_DIR, exist_ok=True)

BEIGE_COLOR = "#f5e6c8"  # soft beige "epithelial skin" color

frame_paths = []

# 1. Collect VTU snapshots from results_from_time_N directories
snapshots = collect_vtu_files(INPUT_DIR)

if not snapshots:
    raise FileNotFoundError(
        f"No VTU snapshots found in results_from_time_* under {INPUT_DIR}"
    )

print(f"Found {len(snapshots)} hourly VTU snapshots")

# 2. Loop through and make beige frames
for frame_idx, (hour, vtu_path) in enumerate(snapshots):
    print(f"[BEIGE] Processing hour={hour}")

    points, cell_polys = load_cells_from_vtu(vtu_path)

    fig, ax = plt.subplots(figsize=(12, 4))

    for poly in cell_polys:
        pts = points[poly]
        ax.fill(
            pts[:, 0],
            pts[:, 1],
            facecolor=BEIGE_COLOR,
            edgecolor="k",
            linewidth=0.3,
            alpha=0.95,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Mesh evolution (t = {hour} h)")

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)

    plt.tight_layout()
    frame_path = os.path.join(GENERIC_FRAME_DIR, f"generic_{frame_idx:04d}.png")
    fig.savefig(frame_path, dpi=150)
    plt.close(fig)

    frame_paths.append(frame_path)

if not frame_paths:
    raise RuntimeError(
        "No frames were generated. Check the VTU collection."
    )

# 3. Make GIF
images = [imageio.imread(p) for p in frame_paths]
imageio.mimsave(GENERIC_GIF, images, fps=15)

print(f"Saved {len(frame_paths)} beige frames to: {GENERIC_FRAME_DIR}")
print(f"Saved generic beige mesh GIF to        : {GENERIC_GIF}")



Found 153 hourly VTU snapshots
[BEIGE] Processing hour=0
[BEIGE] Processing hour=1
[BEIGE] Processing hour=2
[BEIGE] Processing hour=3
[BEIGE] Processing hour=4
[BEIGE] Processing hour=5
[BEIGE] Processing hour=6
[BEIGE] Processing hour=7
[BEIGE] Processing hour=8
[BEIGE] Processing hour=9
[BEIGE] Processing hour=10
[BEIGE] Processing hour=11
[BEIGE] Processing hour=12
[BEIGE] Processing hour=13
[BEIGE] Processing hour=14
[BEIGE] Processing hour=15
[BEIGE] Processing hour=16
[BEIGE] Processing hour=17
[BEIGE] Processing hour=18
[BEIGE] Processing hour=19
[BEIGE] Processing hour=20
[BEIGE] Processing hour=21
[BEIGE] Processing hour=22
[BEIGE] Processing hour=23
[BEIGE] Processing hour=24
[BEIGE] Processing hour=25
[BEIGE] Processing hour=26
[BEIGE] Processing hour=27
[BEIGE] Processing hour=28
[BEIGE] Processing hour=29
[BEIGE] Processing hour=30
[BEIGE] Processing hour=31
[BEIGE] Processing hour=32
[BEIGE] Processing hour=33
[BEIGE] Processing hour=34
[BEIGE] Processing hour=35
[BEIGE]

In [35]:

# ======================================================================
# BLOCK A — STATS & TIME SERIES FOR 60x8 RECTANGLE (C++ Chaste)
# Computes:
#   - cell count vs time
#   - average #vertices per cell vs time
#   - average area per cell vs time
#   - average perimeter per cell vs time
# Also computes global min/max for vertices, area, perimeter (used in Block B)
# ======================================================================

import os
import csv
import numpy as np
import meshio
import matplotlib.pyplot as plt

# Domain bounds (60x8 rectangle — matches C++ simulation)
X_MIN, X_MAX = 0.0, 60.0
Y_MIN, Y_MAX = 0.0, 8.0

# Where to save figures & CSV
STATS_CSV = os.path.join(INPUT_DIR, "rectangle_60x8_cell_stats.csv")
FIG_CELLCOUNT = os.path.join(INPUT_DIR, "rectangle_60x8_cellcount_vs_time.png")
FIG_VERTS = os.path.join(INPUT_DIR, "rectangle_60x8_avg_vertices_vs_time.png")
FIG_AREA = os.path.join(INPUT_DIR, "rectangle_60x8_avg_area_vs_time.png")
FIG_PERIM = os.path.join(INPUT_DIR, "rectangle_60x8_avg_perimeter_vs_time.png")

# ----------------------------------------------------------------------
# Collect VTU files from results_from_time_N directories
# ----------------------------------------------------------------------

snapshots = collect_vtu_files(INPUT_DIR)

print(f"Found {len(snapshots)} hourly VTU snapshots in:\n  {INPUT_DIR}")

if not snapshots:
    raise FileNotFoundError("No VTU snapshots found. Check INPUT_DIR.")

# ----------------------------------------------------------------------
# Loop over VTUs, compute stats
# ----------------------------------------------------------------------

frame_indices = []
hours_list = []
cell_counts = []
avg_vertices = []
avg_areas = []
avg_perims = []

global_min_vertices = np.inf
global_max_vertices = -np.inf
global_min_area = np.inf
global_max_area = -np.inf
global_min_perim = np.inf
global_max_perim = -np.inf

for frame_idx, (hour, vtu_path) in enumerate(snapshots):
    points, cell_polys = load_cells_from_vtu(vtu_path)
    n_cells = len(cell_polys)

    per_cell_vertices = np.array([len(poly) for poly in cell_polys], dtype=float)
    per_cell_areas = np.zeros(n_cells, dtype=float)
    per_cell_perims = np.zeros(n_cells, dtype=float)

    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        A = polygon_area(pts)
        P = polygon_perimeter(pts)
        per_cell_areas[i] = A
        per_cell_perims[i] = P

    frame_indices.append(frame_idx)
    hours_list.append(hour)
    cell_counts.append(n_cells)
    avg_vertices.append(per_cell_vertices.mean())
    avg_areas.append(per_cell_areas.mean())
    avg_perims.append(per_cell_perims.mean())

    # Update global mins/maxes for colorbars later
    global_min_vertices = min(global_min_vertices, per_cell_vertices.min())
    global_max_vertices = max(global_max_vertices, per_cell_vertices.max())

    global_min_area = min(global_min_area, per_cell_areas.min())
    global_max_area = max(global_max_area, per_cell_areas.max())

    global_min_perim = min(global_min_perim, per_cell_perims.min())
    global_max_perim = max(global_max_perim, per_cell_perims.max())

    print(
        f"Frame {frame_idx}: hour={hour}, cells={n_cells}, "
        f"<verts>={avg_vertices[-1]:.2f}, <A>={avg_areas[-1]:.3f}, <P>={avg_perims[-1]:.3f}"
    )

frame_indices = np.array(frame_indices)
hours_arr = np.array(hours_list)
cell_counts = np.array(cell_counts)
avg_vertices = np.array(avg_vertices)
avg_areas = np.array(avg_areas)
avg_perims = np.array(avg_perims)

print("\nGlobal ranges (for GIF colorbars later):")
print(f"  vertices per cell -> min={global_min_vertices}, max={global_max_vertices}")
print(f"  area per cell     -> min={global_min_area:.4f}, max={global_max_area:.4f}")
print(f"  perimeter per cell-> min={global_min_perim:.4f}, max={global_max_perim:.4f}")

# ----------------------------------------------------------------------
# Save stats as CSV
# ----------------------------------------------------------------------

with open(STATS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame_idx",
        "hour",
        "cell_count",
        "avg_vertices",
        "avg_area",
        "avg_perimeter",
    ])
    for i in range(len(frame_indices)):
        writer.writerow([
            int(frame_indices[i]),
            int(hours_arr[i]),
            int(cell_counts[i]),
            float(avg_vertices[i]),
            float(avg_areas[i]),
            float(avg_perims[i]),
        ])

print(f"\nSaved stats CSV to: {STATS_CSV}")

# ----------------------------------------------------------------------
# Plot time-series
# ----------------------------------------------------------------------

# 1) Cell count vs hour
plt.figure(figsize=(8, 4))
plt.plot(hours_arr, cell_counts, marker="o", markersize=2)
plt.xlabel("Simulation time (hours)")
plt.ylabel("Cell count")
plt.title("Number of cells vs time (60x8 rectangle)")
plt.tight_layout()
plt.savefig(FIG_CELLCOUNT, dpi=150)
plt.close()
print(f"Saved cell count figure to: {FIG_CELLCOUNT}")

# 2) Avg vertices vs hour
plt.figure(figsize=(8, 4))
plt.plot(hours_arr, avg_vertices, marker="o", markersize=2)
plt.xlabel("Simulation time (hours)")
plt.ylabel("Average #vertices per cell")
plt.title("Average cell vertices vs time (60x8 rectangle)")
plt.tight_layout()
plt.savefig(FIG_VERTS, dpi=150)
plt.close()
print(f"Saved avg vertices figure to: {FIG_VERTS}")

# 3) Avg area vs hour
plt.figure(figsize=(8, 4))
plt.plot(hours_arr, avg_areas, marker="o", markersize=2)
plt.xlabel("Simulation time (hours)")
plt.ylabel("Average cell area")
plt.title("Average cell area vs time (60x8 rectangle)")
plt.tight_layout()
plt.savefig(FIG_AREA, dpi=150)
plt.close()
print(f"Saved avg area figure to: {FIG_AREA}")

# 4) Avg perimeter vs hour
plt.figure(figsize=(8, 4))
plt.plot(hours_arr, avg_perims, marker="o", markersize=2)
plt.xlabel("Simulation time (hours)")
plt.ylabel("Average cell perimeter")
plt.title("Average cell perimeter vs time (60x8 rectangle)")
plt.tight_layout()
plt.savefig(FIG_PERIM, dpi=150)
plt.close()
print(f"Saved avg perimeter figure to: {FIG_PERIM}")

# Keep global ranges available for Block B
GLOBAL_RANGES = {
    "vertices": (global_min_vertices, global_max_vertices),
    "area": (global_min_area, global_max_area),
    "perimeter": (global_min_perim, global_max_perim),
}
print("\nStored GLOBAL_RANGES for use in Block B.")


Found 153 hourly VTU snapshots in:
  C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50
Frame 0: hour=0, cells=6, <verts>=6.00, <A>=0.913, <P>=3.561
Frame 1: hour=1, cells=6, <verts>=6.00, <A>=0.918, <P>=3.579
Frame 2: hour=2, cells=7, <verts>=5.86, <A>=0.787, <P>=3.355
Frame 3: hour=3, cells=7, <verts>=5.86, <A>=0.799, <P>=3.350
Frame 4: hour=4, cells=9, <verts>=5.67, <A>=0.632, <P>=3.045
Frame 5: hour=5, cells=9, <verts>=5.67, <A>=0.646, <P>=3.043
Frame 6: hour=6, cells=9, <verts>=5.67, <A>=0.672, <P>=3.101
Frame 7: hour=7, cells=9, <verts>=5.67, <A>=0.698, <P>=3.165
Frame 8: hour=8, cells=9, <verts>=5.67, <A>=0.724, <P>=3.227
Frame 9: hour=9, cells=9, <verts>=5.67, <A>=0.749, <P>=3.287
Frame 10: hour=10, cells=9, <verts>=5.67, <A>=0.772, <P>=3.341
Frame 11: hour=11, cells=10, <verts>=5.60, <A>=0.713, <P>=3.224
Frame 12: hour=12, cells=10, <verts>=5.60, <A>=0.736, <P>=3.262
Frame 13: hour=13, cells=10, <verts>=5.60, <A

In [36]:

# ======================================================================
# BLOCK B — COLORED GIFS FOR 60x8 RECTANGLE (C++ Chaste)
# Produces:
#   1) vertices_colored_cells.gif (cells colored by #vertices)
#   2) area_colored_cells.gif     (cells colored by area)
#   3) perimeter_colored_cells.gif (cells colored by perimeter)
# ======================================================================

import os
import numpy as np
import meshio
import matplotlib.pyplot as plt
import imageio.v2 as imageio

from matplotlib.colors import Normalize

# Re-use same INPUT_DIR and X/Y bounds as Block A
# Make sure Block A ran first
print("Using INPUT_DIR:", INPUT_DIR)

VERTEX_FRAME_DIR = os.path.join(INPUT_DIR, "vertex_count_frames")
AREA_FRAME_DIR = os.path.join(INPUT_DIR, "area_frames")
PERIM_FRAME_DIR = os.path.join(INPUT_DIR, "perimeter_frames")

os.makedirs(VERTEX_FRAME_DIR, exist_ok=True)
os.makedirs(AREA_FRAME_DIR, exist_ok=True)
os.makedirs(PERIM_FRAME_DIR, exist_ok=True)

GIF_VERTICES = os.path.join(INPUT_DIR, "vertices_colored_cells.gif")
GIF_AREA = os.path.join(INPUT_DIR, "area_colored_cells.gif")
GIF_PERIM = os.path.join(INPUT_DIR, "perimeter_colored_cells.gif")

# ----------------------------------------------------------------------
# Get VTU snapshots from results_from_time_N directories
# ----------------------------------------------------------------------

snapshots = collect_vtu_files(INPUT_DIR)

print(f"Found {len(snapshots)} hourly VTU snapshots.")

if not snapshots:
    raise FileNotFoundError("No VTU snapshots found. Check INPUT_DIR.")

# ----------------------------------------------------------------------
# Global ranges for colormaps (from Block A)
# ----------------------------------------------------------------------

if "GLOBAL_RANGES" not in globals():
    raise RuntimeError("GLOBAL_RANGES not found. Run Block A first.")

vmin_verts, vmax_verts = GLOBAL_RANGES["vertices"]
vmin_area, vmax_area = GLOBAL_RANGES["area"]
vmin_perim, vmax_perim = GLOBAL_RANGES["perimeter"]

norm_verts = Normalize(vmin=vmin_verts, vmax=vmax_verts)
norm_area = Normalize(vmin=vmin_area, vmax=vmax_area)
norm_perim = Normalize(vmin=vmin_perim, vmax=vmax_perim)

cmap = plt.cm.viridis

vertex_frame_paths = []
area_frame_paths = []
perim_frame_paths = []

# ----------------------------------------------------------------------
# Loop over timesteps, make 3 frames per time
# ----------------------------------------------------------------------

for frame_idx, (hour, vtu_path) in enumerate(snapshots):
    points, cell_polys = load_cells_from_vtu(vtu_path)

    n_cells = len(cell_polys)
    per_cell_vertices = np.array([len(poly) for poly in cell_polys], dtype=float)
    per_cell_areas = np.zeros(n_cells, dtype=float)
    per_cell_perims = np.zeros(n_cells, dtype=float)

    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        per_cell_areas[i] = polygon_area(pts)
        per_cell_perims[i] = polygon_perimeter(pts)

    print(f"Rendering colored frames for hour={hour}, cells={n_cells}")

    # --------------------- 1) VERTEX COUNT GIF -------------------------
    fig, ax = plt.subplots(figsize=(12, 4))
    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        color = cmap(norm_verts(per_cell_vertices[i]))
        ax.fill(
            pts[:, 0],
            pts[:, 1],
            facecolor=color,
            edgecolor="k",
            linewidth=0.3,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_title(f"Vertices per cell (t = {hour} h)")

    sm = plt.cm.ScalarMappable(norm=norm_verts, cmap=cmap)
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', location='bottom')
    cbar.set_label("#vertices")

    plt.tight_layout()
    fpath_v = os.path.join(VERTEX_FRAME_DIR, f"vertices_{frame_idx:04d}.png")
    fig.savefig(fpath_v, dpi=150)
    plt.close(fig)
    vertex_frame_paths.append(fpath_v)

    # --------------------- 2) AREA GIF ------------------------------
    fig, ax = plt.subplots(figsize=(12, 4))
    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        color = cmap(norm_area(per_cell_areas[i]))
        ax.fill(
            pts[:, 0],
            pts[:, 1],
            facecolor=color,
            edgecolor="k",
            linewidth=0.3,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_title(f"Area per cell (t = {hour} h)")

    sm = plt.cm.ScalarMappable(norm=norm_area, cmap=cmap)
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', location='bottom')
    cbar.set_label("Area")

    plt.tight_layout()
    fpath_a = os.path.join(AREA_FRAME_DIR, f"area_{frame_idx:04d}.png")
    fig.savefig(fpath_a, dpi=150)
    plt.close(fig)
    area_frame_paths.append(fpath_a)

    # --------------------- 3) PERIMETER GIF -------------------------
    fig, ax = plt.subplots(figsize=(12, 4))
    for i, poly in enumerate(cell_polys):
        pts = points[poly]
        color = cmap(norm_perim(per_cell_perims[i]))
        ax.fill(
            pts[:, 0],
            pts[:, 1],
            facecolor=color,
            edgecolor="k",
            linewidth=0.3,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_title(f"Perimeter per cell (t = {hour} h)")

    sm = plt.cm.ScalarMappable(norm=norm_perim, cmap=cmap)
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', location='bottom')
    cbar.set_label("Perimeter")

    plt.tight_layout()
    fpath_p = os.path.join(PERIM_FRAME_DIR, f"perim_{frame_idx:04d}.png")
    fig.savefig(fpath_p, dpi=150)
    plt.close(fig)
    perim_frame_paths.append(fpath_p)

# ----------------------------------------------------------------------
# Build GIFs
# ----------------------------------------------------------------------

if not vertex_frame_paths:
    raise RuntimeError("No vertex frames generated; check VTU loop.")

images_v = [imageio.imread(p) for p in vertex_frame_paths]
imageio.mimsave(GIF_VERTICES, images_v, fps=10)
print(f"\nSaved vertices-colored GIF to: {GIF_VERTICES}")

if not area_frame_paths:
    raise RuntimeError("No area frames generated; check VTU loop.")

images_a = [imageio.imread(p) for p in area_frame_paths]
imageio.mimsave(GIF_AREA, images_a, fps=10)
print(f"Saved area-colored GIF to     : {GIF_AREA}")

if not perim_frame_paths:
    raise RuntimeError("No perimeter frames generated; check VTU loop.")

images_p = [imageio.imread(p) for p in perim_frame_paths]
imageio.mimsave(GIF_PERIM, images_p, fps=10)
print(f"Saved perimeter-colored GIF to: {GIF_PERIM}")



Using INPUT_DIR: C:\Users\cmchu\OneDrive\Desktop\Gu Lab\BIOE242 Data\testoutput\EpithelialRheology\Proliferation_test\CycleScale_1.50
Found 153 hourly VTU snapshots.
Rendering colored frames for hour=0, cells=6
Rendering colored frames for hour=1, cells=6
Rendering colored frames for hour=2, cells=7
Rendering colored frames for hour=3, cells=7
Rendering colored frames for hour=4, cells=9
Rendering colored frames for hour=5, cells=9
Rendering colored frames for hour=6, cells=9
Rendering colored frames for hour=7, cells=9
Rendering colored frames for hour=8, cells=9
Rendering colored frames for hour=9, cells=9
Rendering colored frames for hour=10, cells=9
Rendering colored frames for hour=11, cells=10
Rendering colored frames for hour=12, cells=10
Rendering colored frames for hour=13, cells=10
Rendering colored frames for hour=14, cells=10
Rendering colored frames for hour=15, cells=10
Rendering colored frames for hour=16, cells=10
Rendering colored frames for hour=17, cells=10
Rendering